Cell 1：清理并 clone 官方 COSMOS

In [1]:
# ============================================================
# Cell 1
# Clean workspace + clone official COSMOS
# ============================================================

from pathlib import Path
import shutil
import subprocess
import sys


WORK_ROOT = Path(
    "/kaggle/working"
)

COSMOS_ROOT = (
    WORK_ROOT
    / "COSMOS"
)

COSMOS_OUTPUT_ROOT = (
    WORK_ROOT
    / "COSMOS_baseline"
)

DATA_ROOT = Path(
    "/kaggle/input/datasets/wuvdji/smgc-data"
)


print("=" * 100)
print("RESET COSMOS WORKSPACE")
print("=" * 100)


if COSMOS_ROOT.exists():

    shutil.rmtree(
        COSMOS_ROOT
    )

    print(
        "Deleted old repo:",
        COSMOS_ROOT
    )


if COSMOS_OUTPUT_ROOT.exists():

    shutil.rmtree(
        COSMOS_OUTPUT_ROOT
    )

    print(
        "Deleted old outputs:",
        COSMOS_OUTPUT_ROOT
    )


# Clear stale modules
for name in list(sys.modules):

    if (
        name == "COSMOS"
        or name.startswith("COSMOS.")
    ):

        del sys.modules[name]


# Clone official source
subprocess.run(
    [
        "git",
        "clone",
        "https://github.com/Lin-Xu-lab/COSMOS.git",
        str(COSMOS_ROOT),
    ],
    check=True,
)


assert COSMOS_ROOT.exists()
assert DATA_ROOT.exists()


COSMOS_COMMIT = (
    subprocess.check_output(
        [
            "git",
            "-C",
            str(COSMOS_ROOT),
            "rev-parse",
            "HEAD",
        ],
        text=True,
    )
    .strip()
)


print(
    "\nCOSMOS root:",
    COSMOS_ROOT
)

print(
    "Data root  :",
    DATA_ROOT
)

print(
    "Git commit :",
    COSMOS_COMMIT
)

print(
    "\nPASS: COSMOS repository ready."
)

RESET COSMOS WORKSPACE


Cloning into '/kaggle/working/COSMOS'...



COSMOS root: /kaggle/working/COSMOS
Data root  : /kaggle/input/datasets/wuvdji/smgc-data
Git commit : 56ea355be51e64d9253e2871b8bd447fdfd0d230

PASS: COSMOS repository ready.


Cell 2：安装依赖

In [3]:
# ============================================================
# Cell 2 — REVISED
# Install COSMOS dependencies safely on Kaggle Python 3.12
#
# IMPORTANT:
# Do NOT pip install the legacy "louvain" package.
# For RNA+ATAC downstream Louvain clustering we will use
# python-igraph's built-in Louvain implementation.
# ============================================================

import sys
import subprocess
import importlib.util

print("=" * 100)
print("INSTALL COSMOS DEPENDENCIES — PYTHON 3.12 COMPATIBLE")
print("=" * 100)


def ensure_package(
    module_name,
    pip_name,
):
    if importlib.util.find_spec(
        module_name
    ) is not None:

        print(
            f"[OK] {module_name} already installed"
        )

        return

    print(
        f"[INSTALL] {pip_name}"
    )

    subprocess.run(
        [
            sys.executable,
            "-m",
            "pip",
            "install",
            "--prefer-binary",
            pip_name,
        ],
        check=True,
    )


# ------------------------------------------------------------
# Core versions already used successfully in our benchmark env
# ------------------------------------------------------------

subprocess.run(
    [
        sys.executable,
        "-m",
        "pip",
        "install",
        "--prefer-binary",
        "anndata==0.11.4",
        "scanpy==1.11.4",
    ],
    check=True,
)


# ------------------------------------------------------------
# COSMOS runtime dependencies
# ------------------------------------------------------------

ensure_package(
    "gudhi",
    "gudhi",
)

ensure_package(
    "cmcrameri",
    "cmcrameri",
)

ensure_package(
    "umap",
    "umap-learn",
)

ensure_package(
    "igraph",
    "igraph",
)

ensure_package(
    "leidenalg",
    "leidenalg",
)

ensure_package(
    "torch_geometric",
    "torch-geometric==2.8.0.post1",
)

ensure_package(
    "networkx",
    "networkx",
)


print(
    "\nPASS: COSMOS dependencies installed."
)

print(
    "NOTE: legacy PyPI 'louvain' package intentionally NOT installed."
)

INSTALL COSMOS DEPENDENCIES — PYTHON 3.12 COMPATIBLE
  Using cached anndata-0.11.4-py3-none-any.whl.metadata (9.3 kB)
  Using cached scanpy-1.11.4-py3-none-any.whl.metadata (9.2 kB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 144.5/144.5 kB 3.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.1/2.1 MB 36.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 79.1/79.1 kB 4.4 MB/s eta 0:00:00
[INSTALL] gudhi
  Using cached gudhi-3.13.0-cp312-cp312-manylinux_2_27_x86_64.manylinux_2_28_x86_64.whl.metadata (2.3 kB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.5/4.5 MB 28.0 MB/s eta 0:00:00
[INSTALL] cmcrameri
  Using cached cmcrameri-1.10-py3-none-any.whl.metadata (5.2 kB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 277.7/277.7 kB 6.4 MB/s eta 0:00:00
[OK] umap already installed
[OK] igraph already installed
[INSTALL] leidenalg
  Using cached leidenalg-0.12.0-cp38-abi3-manylinux_2_27_x86_64.manylinux_2_28_x86_64.whl.metadata (10 kB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

Cell 3：兼容补丁 + 环境审

In [4]:
# ============================================================
# Cell 3 — REVISED
# COSMOS environment + compatibility audit
# ============================================================

import sys
import importlib
import inspect

import numpy as np
import scipy
import scipy.sparse as sp
import pandas as pd
import torch
import scanpy as sc
import anndata
import sklearn
import torch_geometric
import igraph as ig
import leidenalg
import gudhi
import cmcrameri


# ------------------------------------------------------------
# SciPy sparse compatibility
# ------------------------------------------------------------

if not hasattr(
    sp.spmatrix,
    "A",
):

    sp.spmatrix.A = property(
        lambda self:
            self.toarray()
    )


for cls in [
    sp.csr_matrix,
    sp.csc_matrix,
    sp.coo_matrix,
]:

    if not hasattr(
        cls,
        "A",
    ):

        cls.A = property(
            lambda self:
                self.toarray()
        )


# ------------------------------------------------------------
# Official COSMOS repo first
# ------------------------------------------------------------

repo_path = str(
    COSMOS_ROOT
)

sys.path = [
    p
    for p in sys.path
    if p != repo_path
]

sys.path.insert(
    0,
    repo_path,
)

importlib.invalidate_caches()


# ------------------------------------------------------------
# COSMOS imports
# ------------------------------------------------------------

from COSMOS import cosmos
from COSMOS.pyWNN import pyWNN


print("=" * 100)
print("COSMOS ENVIRONMENT AUDIT")
print("=" * 100)

print(
    "Python         :",
    sys.version.split()[0]
)

print(
    "PyTorch        :",
    torch.__version__
)

print(
    "CUDA runtime   :",
    torch.version.cuda
)

print(
    "CUDA available :",
    torch.cuda.is_available()
)

if torch.cuda.is_available():

    print(
        "GPU            :",
        torch.cuda.get_device_name(0)
    )


print(
    "NumPy          :",
    np.__version__
)

print(
    "SciPy          :",
    scipy.__version__
)

print(
    "Scanpy         :",
    sc.__version__
)

print(
    "AnnData        :",
    anndata.__version__
)

print(
    "sklearn        :",
    sklearn.__version__
)

print(
    "PyG            :",
    torch_geometric.__version__
)

print(
    "igraph         :",
    ig.__version__
)

print(
    "leidenalg      :",
    getattr(
        leidenalg,
        "__version__",
        "unknown",
    )
)


print(
    "\nCosmos.preprocessing_data:"
)

print(
    inspect.signature(
        cosmos.Cosmos.preprocessing_data
    )
)


print(
    "\nCosmos.train:"
)

print(
    inspect.signature(
        cosmos.Cosmos.train
    )
)


# ------------------------------------------------------------
# Verify modern igraph Louvain supports resolution
# ------------------------------------------------------------

louvain_signature = (
    inspect.signature(
        ig.Graph.community_multilevel
    )
)

print(
    "\nigraph Louvain:",
    louvain_signature
)


assert (
    "resolution"
    in louvain_signature.parameters
), (
    "Installed igraph does not expose "
    "resolution for community_multilevel"
)


assert torch.cuda.is_available()


print(
    "\nPASS: COSMOS imports successfully."
)

print(
    "PASS: igraph Louvain with resolution is available."
)

print(
    "PASS: COSMOS environment ready."
)

COSMOS ENVIRONMENT AUDIT
Python         : 3.12.13
PyTorch        : 2.10.0+cu128
CUDA runtime   : 12.8
CUDA available : True
GPU            : Tesla T4
NumPy          : 2.0.2
SciPy          : 1.16.3
Scanpy         : 1.11.4
AnnData        : 0.11.4
sklearn        : 1.6.1
PyG            : 2.8.0.post1
igraph         : 1.0.0
leidenalg      : 0.12.0

Cosmos.preprocessing_data:
(self, do_norm=False, do_log=False, n_top_genes=None, do_pca=False, n_neighbors=10)

Cosmos.train:
(self, embedding_save_filepath='./embedding.tsv', weights_save_filepath='./weights.tsv', spatial_regularization_strength=0.05, z_dim=50, lr=0.001, wnn_epoch=100, total_epoch=1000, max_patience_bef=10, max_patience_aft=30, min_stop=100, random_seed=42, gpu=0, regularization_acceleration=True, edge_subset_sz=1000000)

igraph Louvain: (graph, weights=None, return_levels=False, resolution=1)

PASS: COSMOS imports successfully.
PASS: igraph Louvain with resolution is available.
PASS: COSMOS environment ready.


/kaggle/working/COSMOS/COSMOS/modulesWNN.py:37: SyntaxWarning: invalid escape sequence '\m'
  paper based on user-defined encoder and summary model :math:`\mathcal{E}`


Cell 4：5 数据集 loader + COSMOS 官方模态预处理

In [5]:
# ============================================================
# Cell 4
# Dataset loader + modality-specific COSMOS preprocessing
# ============================================================

import numpy as np
import scanpy as sc


DATASET_SPECS = {

    "HLN-A1": {

        "rna":
            DATA_ROOT
            / "Human_Lymph_Nodes/A1/adata_RNA.h5ad",

        "second":
            DATA_ROOT
            / "Human_Lymph_Nodes/A1/adata_ADT.h5ad",

        "second_type":
            "ADT",

        "K":
            10,

        "n_spots":
            3484,
    },


    "HLN-D1": {

        "rna":
            DATA_ROOT
            / "Human_Lymph_Nodes/D1/adata_RNA.h5ad",

        "second":
            DATA_ROOT
            / "Human_Lymph_Nodes/D1/adata_ADT.h5ad",

        "second_type":
            "ADT",

        "K":
            11,

        "n_spots":
            3359,
    },


    "E18.5": {

        "rna":
            DATA_ROOT
            / "E18.5_mouse_brain/adata_RNA.h5ad",

        "second":
            DATA_ROOT
            / "E18.5_mouse_brain/adata_ATAC.h5ad",

        "second_type":
            "ATAC",

        "K":
            14,

        "n_spots":
            2129,
    },


    "S2-E15": {

        "rna":
            DATA_ROOT
            / "Mouse_Embryos_S2/E15/adata_RNA.h5ad",

        "second":
            DATA_ROOT
            / "Mouse_Embryos_S2/E15/adata_ATAC.h5ad",

        "second_type":
            "ATAC",

        "K":
            15,

        "n_spots":
            1939,
    },


    "S2-E18": {

        "rna":
            DATA_ROOT
            / "Mouse_Embryos_S2/E18/adata_RNA.h5ad",

        "second":
            DATA_ROOT
            / "Mouse_Embryos_S2/E18/adata_ATAC.h5ad",

        "second_type":
            "ATAC",

        "K":
            16,

        "n_spots":
            2248,
    },
}


def load_cosmos_dataset(
    dataset_name,
):

    spec = DATASET_SPECS[
        dataset_name
    ]


    assert spec["rna"].exists(), (
        spec["rna"]
    )

    assert spec["second"].exists(), (
        spec["second"]
    )


    rna = sc.read_h5ad(
        spec["rna"]
    )

    second = sc.read_h5ad(
        spec["second"]
    )


    rna.var_names_make_unique()
    second.var_names_make_unique()


    assert (
        rna.n_obs
        == spec["n_spots"]
    )

    assert (
        second.n_obs
        == spec["n_spots"]
    )


    assert np.array_equal(
        np.asarray(
            rna.obs_names
        ),
        np.asarray(
            second.obs_names
        ),
    )


    # ========================================================
    # E18.5 metadata adapter
    # ========================================================

    if dataset_name == "E18.5":

        coords = np.column_stack(
            [
                np.asarray(
                    rna.obs[
                        "array_col"
                    ],
                    dtype=np.float64,
                ),

                np.asarray(
                    rna.obs[
                        "array_row"
                    ],
                    dtype=np.float64,
                ),
            ]
        )


        labels = (
            rna.obs[
                "Combined_Clusters_annotation"
            ]
            .astype(str)
            .to_numpy()
        )


        for adata in [
            rna,
            second,
        ]:

            adata.obsm[
                "spatial"
            ] = coords.copy()

            adata.obs[
                "Spatial_Label"
            ] = labels.copy()


    # ========================================================
    # Structural audit
    # ========================================================

    for adata in [
        rna,
        second,
    ]:

        assert (
            "Spatial_Label"
            in adata.obs.columns
        )

        assert (
            "spatial"
            in adata.obsm
        )


    assert (
        rna.obs[
            "Spatial_Label"
        ].nunique()
        == spec["K"]
    )


    assert np.allclose(
        np.asarray(
            rna.obsm[
                "spatial"
            ]
        ),
        np.asarray(
            second.obsm[
                "spatial"
            ]
        ),
    )


    # ========================================================
    # Official modality-specific preprocessing
    # ========================================================

    if (
        spec["second_type"]
        == "ADT"
    ):

        # Official COSMOS RNA+Protein tutorial:
        # RNA normalize_per_cell + log1p
        # Protein log1p

        sc.pp.normalize_per_cell(
            rna
        )

        sc.pp.log1p(
            rna
        )

        sc.pp.log1p(
            second
        )


        spatial_reg = 0.05
        cluster_method = "leiden"


    elif (
        spec["second_type"]
        == "ATAC"
    ):

        # Official COSMOS RNA+ATAC tutorial:
        # no additional normalization/log transformation
        # before COSMOS.preprocessing_data()

        spatial_reg = 0.01
        cluster_method = "louvain"


    else:

        raise ValueError(
            spec["second_type"]
        )


    print(
        f"{dataset_name}: "
        f"{spec['n_spots']} spots | "
        f"K={spec['K']} | "
        f"RNA + {spec['second_type']} | "
        f"spatial_reg={spatial_reg} | "
        f"{cluster_method}"
    )


    return (
        rna,
        second,
        spec,
        spatial_reg,
        cluster_method,
    )


print(
    "PASS: COSMOS dataset loader defined."
)

PASS: COSMOS dataset loader defined.


Cell 5：官方 embedding readout + 目标 K resolution search

In [6]:
# ============================================================
# Cell 5 — REVISED
# COSMOS downstream clustering
#
# RNA+ADT:
#   Leiden via Scanpy / leidenalg
#
# RNA+ATAC:
#   Louvain via python-igraph community_multilevel
#
# Resolution selected ONLY by target K.
# GT / ARI / NMI are never used in resolution selection.
#
# Compatibility note:
# Legacy PyPI "louvain" has no CPython 3.12 wheel.
# We therefore retain the Louvain algorithm using modern
# python-igraph's built-in implementation.
# ============================================================

import random

import numpy as np
import scipy.sparse as sp
import anndata as ad
import scanpy as sc
import igraph as ig


CLUSTER_RANDOM_STATE = 0


def _build_igraph_from_scanpy_neighbors(
    adata_emb,
):

    adjacency = (
        adata_emb
        .obsp[
            "connectivities"
        ]
        .tocsr()
    )


    # Scanpy connectivities should already be symmetric.
    # Enforce symmetry explicitly for igraph Louvain.
    adjacency = (
        adjacency.maximum(
            adjacency.T
        )
    )


    # One undirected edge per pair.
    upper = sp.triu(
        adjacency,
        k=1,
    ).tocoo()


    edges = list(
        zip(
            upper.row.tolist(),
            upper.col.tolist(),
        )
    )


    graph = ig.Graph(
        n=adata_emb.n_obs,
        edges=edges,
        directed=False,
    )


    assert (
        graph.vcount()
        == adata_emb.n_obs
    )


    assert (
        graph.ecount()
        > 0
    )


    return graph


def cosmos_cluster_target_k(
    embedding,
    target_k,
    method,
    n_neighbors=50,
    res_start=0.05,
    res_end=2.50,
    res_step=0.01,
):

    embedding = np.asarray(
        embedding,
        dtype=np.float32,
    )


    assert (
        embedding.ndim
        == 2
    )

    assert np.isfinite(
        embedding
    ).all()


    adata_emb = ad.AnnData(
        X=embedding
    )


    # --------------------------------------------------------
    # Same embedding-neighbor construction for both methods
    # --------------------------------------------------------

    sc.pp.neighbors(
        adata_emb,
        n_neighbors=n_neighbors,
        use_rep="X",
        random_state=CLUSTER_RANDOM_STATE,
    )


    # Louvain needs an igraph representation.
    if method == "louvain":

        ig_graph = (
            _build_igraph_from_scanpy_neighbors(
                adata_emb
            )
        )

    else:

        ig_graph = None


    exact_hits = []


    resolutions = np.arange(
        res_start,
        res_end + 1e-12,
        res_step,
    )


    for res in resolutions:

        res = float(
            np.round(
                res,
                8,
            )
        )


        # ====================================================
        # RNA+ADT: Leiden
        # ====================================================

        if method == "leiden":

            sc.tl.leiden(
                adata_emb,
                resolution=res,
                random_state=CLUSTER_RANDOM_STATE,
                key_added="cosmos_cluster",
            )


            labels = (
                adata_emb.obs[
                    "cosmos_cluster"
                ]
                .astype(str)
                .to_numpy()
            )


        # ====================================================
        # RNA+ATAC: Louvain
        # ====================================================

        elif method == "louvain":

            # Reset RNG on every resolution so resolution scans
            # do not inherit RNG state from previous candidates.
            ig.set_random_number_generator(
                random.Random(
                    CLUSTER_RANDOM_STATE
                )
            )


            partition = (
                ig_graph.community_multilevel(
                    weights=None,
                    return_levels=False,
                    resolution=res,
                )
            )


            labels = np.asarray(
                partition.membership,
                dtype=str,
            )


        else:

            raise ValueError(
                f"Unknown clustering method: "
                f"{method}"
            )


        n_cluster = len(
            np.unique(
                labels
            )
        )


        if (
            n_cluster
            == target_k
        ):

            exact_hits.append(
                (
                    res,
                    labels.copy(),
                )
            )


    # --------------------------------------------------------
    # Require exact benchmark K
    # --------------------------------------------------------

    if len(
        exact_hits
    ) == 0:

        raise RuntimeError(
            f"COSMOS {method}: "
            f"no resolution in "
            f"[{res_start}, {res_end}] "
            f"produced target K="
            f"{target_k}"
        )


    # Match our frozen rule:
    # scan upward and retain the LAST target-K resolution.
    selected_res, pred = (
        exact_hits[-1]
    )


    assert (
        len(
            np.unique(
                pred
            )
        )
        == target_k
    )


    print(
        f"COSMOS clustering: "
        f"{method} | "
        f"K={target_k} | "
        f"resolution="
        f"{selected_res:.2f} | "
        f"exact resolutions="
        f"{len(exact_hits)}"
    )


    if method == "louvain":

        print(
            "Louvain backend : "
            "python-igraph "
            "community_multilevel "
            "(Python 3.12 compatibility)"
        )

    else:

        print(
            "Leiden backend  : "
            "Scanpy/leidenalg"
        )


    return (
        pred,
        selected_res,
    )


print(
    "PASS: COSMOS clustering function defined."
)

PASS: COSMOS clustering function defined.


Cell 6：最终单次 runner

In [8]:
# ============================================================
# Cell 6
# COSMOS single-run benchmark runner
# ============================================================

from pathlib import Path
import json
import gc
import time

import numpy as np
import torch

from sklearn.metrics import (
    adjusted_rand_score,
    normalized_mutual_info_score,
)


COSMOS_OUTPUT_ROOT = Path(
    "/kaggle/working/COSMOS_baseline"
)


def dataset_folder_name(
    dataset_name,
):

    return {

        "HLN-A1": "HLNA1",
        "HLN-D1": "HLND1",
        "E18.5": "E185",
        "S2-E15": "S2E15",
        "S2-E18": "S2E18",

    }[
        dataset_name
    ]


def run_cosmos_once(
    dataset_name,
    seed,
    stage="smoke",
):

    (
        rna,
        second,
        spec,
        spatial_reg,
        cluster_method,
    ) = load_cosmos_dataset(
        dataset_name
    )


    print(
        "\n" + "=" * 110
    )

    print(
        f"COSMOS | "
        f"{dataset_name} | "
        f"seed={seed}"
    )

    print(
        "=" * 110
    )


    print(
        "Input spots :",
        spec["n_spots"]
    )

    print(
        "Target K    :",
        spec["K"]
    )

    print(
        "Modality    :",
        f"RNA + {spec['second_type']}"
    )

    print(
        "Cluster     :",
        cluster_method
    )

    print(
        "Spatial reg :",
        spatial_reg
    )


    # ========================================================
    # COSMOS object
    # ========================================================

    model = cosmos.Cosmos(
        adata1=rna,
        adata2=second,
    )


    # Official tutorial
    model.preprocessing_data(
        n_neighbors=10
    )


    start_time = time.time()


    # ========================================================
    # Official tutorial training parameters
    # ========================================================

    embedding = model.train(

        spatial_regularization_strength=
            spatial_reg,

        z_dim=50,

        lr=1e-3,

        wnn_epoch=500,

        total_epoch=1000,

        max_patience_bef=10,

        max_patience_aft=30,

        min_stop=200,

        random_seed=int(
            seed
        ),

        gpu=0,

        regularization_acceleration=True,

        edge_subset_sz=1000000,
    )


    elapsed_train = (
        time.time()
        - start_time
    )


    embedding = np.asarray(
        embedding
    )


    weights = np.asarray(
        model.weights
    )


    # ========================================================
    # Structural audit
    # ========================================================

    assert (
        embedding.shape
        == (
            spec["n_spots"],
            50,
        )
    ), embedding.shape


    assert (
        weights.shape
        == (
            spec["n_spots"],
            2,
        )
    ), weights.shape


    assert np.isfinite(
        embedding
    ).all()


    assert np.isfinite(
        weights
    ).all()


    # ========================================================
    # Official-style clustering
    # ========================================================

    pred, resolution = (
        cosmos_cluster_target_k(

            embedding=embedding,

            target_k=spec["K"],

            method=cluster_method,

            n_neighbors=50,
        )
    )


    gt = (
        rna.obs[
            "Spatial_Label"
        ]
        .astype(str)
        .to_numpy()
    )


    coords = np.asarray(
        rna.obsm[
            "spatial"
        ]
    )


    spot_ids = np.asarray(
        rna.obs_names.astype(str)
    )


    assert (
        len(pred)
        == spec["n_spots"]
    )


    assert (
        len(
            np.unique(pred)
        )
        == spec["K"]
    )


    # ========================================================
    # Unified benchmark metrics
    # ========================================================

    ari = adjusted_rand_score(
        gt,
        pred,
    )


    nmi = (
        normalized_mutual_info_score(
            gt,
            pred,
            average_method="max",
        )
    )


    # ========================================================
    # Output
    # ========================================================

    out_dir = (
        COSMOS_OUTPUT_ROOT
        / stage
        / (
            dataset_folder_name(
                dataset_name
            )
            + f"_seed{seed}"
        )
    )


    out_dir.mkdir(
        parents=True,
        exist_ok=True,
    )


    np.save(
        out_dir
        / "embedding.npy",
        embedding,
    )

    np.save(
        out_dir
        / "modality_weights.npy",
        weights,
    )

    np.save(
        out_dir
        / "pred_labels.npy",
        pred,
    )

    np.save(
        out_dir
        / "gt_labels.npy",
        gt,
    )

    np.save(
        out_dir
        / "coords.npy",
        coords,
    )

    np.save(
        out_dir
        / "spot_ids.npy",
        spot_ids,
    )


    metrics = {

        "method":
            "COSMOS",

        "dataset":
            dataset_name,

        "training_seed":
            int(seed),

        "n_spots":
            int(
                spec["n_spots"]
            ),

        "target_K":
            int(
                spec["K"]
            ),

        "predicted_K":
            int(
                len(
                    np.unique(
                        pred
                    )
                )
            ),

        "modalities":
            (
                "RNA+"
                + spec[
                    "second_type"
                ]
            ),

        "spatial_neighbors":
            10,

        "spatial_regularization_strength":
            float(
                spatial_reg
            ),

        "z_dim":
            50,

        "lr":
            0.001,

        "wnn_epoch":
            500,

        "total_epoch":
            1000,

        "max_patience_bef":
            10,

        "max_patience_aft":
            30,

        "min_stop":
            200,

        "cluster_method":
            cluster_method,

        "cluster_neighbors":
            50,

        "cluster_resolution":
            float(
                resolution
            ),

        "cluster_random_state":
            0,

        "ARI":
            float(
                ari
            ),

        "NMI":
            float(
                nmi
            ),

        "NMI_average_method":
            "max",

        "training_seconds":
            float(
                elapsed_train
            ),

        "COSMOS_git_commit":
            COSMOS_COMMIT,
    }


    with (
        out_dir
        / "metrics.json"
    ).open(
        "w",
        encoding="utf-8",
    ) as f:

        json.dump(
            metrics,
            f,
            indent=2,
        )


    print(
        "\n" + "=" * 110
    )

    print(
        f"{dataset_name} COSMOS RESULT"
    )

    print(
        "=" * 110
    )


    print(
        "Embedding shape:",
        embedding.shape
    )

    print(
        "Weights shape  :",
        weights.shape
    )

    print(
        "Predicted K    :",
        len(
            np.unique(pred)
        )
    )

    print(
        "Resolution     :",
        resolution
    )

    print(
        f"ARI = {ari:.12f}"
    )

    print(
        f"NMI = {nmi:.12f}"
    )

    print(
        f"Training runtime = "
        f"{elapsed_train:.2f}s"
    )

    print(
        "\nSaved:",
        out_dir
    )

    print(
        "\nPASS: COSMOS run completed."
    )

    print(
        f"PASS: "
        f"{spec['n_spots']}/"
        f"{spec['n_spots']} "
        f"spots evaluated."
    )


    del model
    del rna
    del second

    gc.collect()

    if torch.cuda.is_available():

        torch.cuda.empty_cache()


    return metrics

Cell 7：HLN-A1 seed0 smoke

In [9]:
# ============================================================
# Cell 7
# COSMOS HLN-A1 seed0 smoke
# ============================================================

HLNA1_SMOKE = run_cosmos_once(
    dataset_name="HLN-A1",
    seed=0,
    stage="smoke",
)


print(
    "\n" + "=" * 100
)

print(
    "HLN-A1 COSMOS SMOKE COMPLETE"
)

print(
    "=" * 100
)

print(
    "ARI:",
    HLNA1_SMOKE[
        "ARI"
    ]
)

print(
    "NMI:",
    HLNA1_SMOKE[
        "NMI"
    ]
)

/usr/local/lib/python3.12/dist-packages/anndata/_core/anndata.py:1758: UserWarning: Variable names are not unique. To make them unique, call `.var_names_make_unique`.
  utils.warn_names_duplicates("var")
/usr/local/lib/python3.12/dist-packages/anndata/_core/anndata.py:1758: UserWarning: Variable names are not unique. To make them unique, call `.var_names_make_unique`.
  utils.warn_names_duplicates("var")
/tmp/ipykernel_58/2485005381.py:273: FutureWarning: Use sc.pp.normalize_total instead
  sc.pp.normalize_per_cell(
/usr/local/lib/python3.12/dist-packages/scanpy/preprocessing/_simple.py:591: FutureWarning: Use sc.pp.normalize_total instead
  normalize_per_cell(


HLN-A1: 3484 spots | K=10 | RNA + ADT | spatial_reg=0.05 | leiden

COSMOS | HLN-A1 | seed=0
Input spots : 3484
Target K    : 10
Modality    : RNA + ADT
Cluster     : leiden
Spatial reg : 0.05
Epoch 1/1000, Loss: 1.4408732652664185
Epoch 11/1000, Loss: 1.4229328632354736
Epoch 21/1000, Loss: 1.416532278060913
Epoch 31/1000, Loss: 1.4125781059265137
Epoch 41/1000, Loss: 1.4097574949264526
Epoch 51/1000, Loss: 1.4075887203216553
Epoch 61/1000, Loss: 1.4039442539215088
Epoch 71/1000, Loss: 1.3976298570632935
Epoch 81/1000, Loss: 1.3855658769607544
Epoch 91/1000, Loss: 1.3400756120681763
Epoch 101/1000, Loss: 1.0974620580673218
Epoch 111/1000, Loss: 0.7427215576171875
Epoch 121/1000, Loss: 0.543057918548584
Epoch 131/1000, Loss: 0.28921473026275635
Epoch 141/1000, Loss: 0.15330007672309875
Epoch 151/1000, Loss: 0.08619809150695801
Epoch 161/1000, Loss: 0.07433197647333145
Epoch 171/1000, Loss: 0.053901612758636475
Epoch 181/1000, Loss: 0.05219266563653946
Epoch 191/1000, Loss: 0.04960485547

/tmp/ipykernel_58/2606605513.py:174: FutureWarning: In the future, the default backend for leiden will be igraph instead of leidenalg.

 To achieve the future defaults please pass: flavor="igraph" and n_iterations=2.  directed must also be False to work with igraph's implementation.
  sc.tl.leiden(


COSMOS clustering: leiden | K=10 | resolution=0.62 | exact resolutions=16
Leiden backend  : Scanpy/leidenalg

HLN-A1 COSMOS RESULT
Embedding shape: (3484, 50)
Weights shape  : (3484, 2)
Predicted K    : 10
Resolution     : 0.62
ARI = 0.169200294479
NMI = 0.280178823260
Training runtime = 102.30s

Saved: /kaggle/working/COSMOS_baseline/smoke/HLNA1_seed0

PASS: COSMOS run completed.
PASS: 3484/3484 spots evaluated.

HLN-A1 COSMOS SMOKE COMPLETE
ARI: 0.16920029447878215
NMI: 0.2801788232601314


Cell 8：E18.5 seed0 smoke

In [10]:
# ============================================================
# Cell 8
# COSMOS E18.5 seed0 smoke
# ============================================================

E185_SMOKE = run_cosmos_once(
    dataset_name="E18.5",
    seed=0,
    stage="smoke",
)


print(
    "\n" + "=" * 100
)

print(
    "E18.5 COSMOS SMOKE COMPLETE"
)

print(
    "=" * 100
)

print(
    "ARI:",
    E185_SMOKE[
        "ARI"
    ]
)

print(
    "NMI:",
    E185_SMOKE[
        "NMI"
    ]
)

E18.5: 2129 spots | K=14 | RNA + ATAC | spatial_reg=0.01 | louvain

COSMOS | E18.5 | seed=0
Input spots : 2129
Target K    : 14
Modality    : RNA + ATAC
Cluster     : louvain
Spatial reg : 0.01
Epoch 1/1000, Loss: 1.4193825721740723
Epoch 11/1000, Loss: 1.4022901058197021
Epoch 21/1000, Loss: 1.3929266929626465
Epoch 31/1000, Loss: 1.3896723985671997
Epoch 41/1000, Loss: 1.3886537551879883
Epoch 51/1000, Loss: 1.3874651193618774
Epoch 61/1000, Loss: 1.3861265182495117
Epoch 71/1000, Loss: 1.3837757110595703
Epoch 81/1000, Loss: 1.3788340091705322
Epoch 91/1000, Loss: 1.3681083917617798
Epoch 101/1000, Loss: 1.3252894878387451
Epoch 111/1000, Loss: 1.1133872270584106
Epoch 121/1000, Loss: 0.8489866852760315
Epoch 131/1000, Loss: 0.5678299069404602
Epoch 141/1000, Loss: 0.338186115026474
Epoch 151/1000, Loss: 0.13207697868347168
Epoch 161/1000, Loss: 0.0639982521533966
Epoch 171/1000, Loss: 0.03938254341483116
Epoch 181/1000, Loss: 0.03265455737709999
Epoch 191/1000, Loss: 0.022774621844

/kaggle/working/COSMOS/COSMOS/pyWNN.py:214: RuntimeWarning: overflow encountered in exp
  self.weights.append( 1 / (1+ np.exp(affinity_ratios[1]-affinity_ratios[0])) )


2000 out of 2129 7.60 seconds elapsed
Selecting top K neighbors
Epoch 261/1000, Loss: 0.061670467257499695
Epoch 271/1000, Loss: 0.028919391334056854
Epoch 281/1000, Loss: 0.019680937752127647
Epoch 291/1000, Loss: 0.012393098324537277
Epoch 301/1000, Loss: 0.00996621698141098
Epoch 311/1000, Loss: 0.009211987257003784
Epoch 321/1000, Loss: 0.011751901358366013
Epoch 331/1000, Loss: 0.00903679896146059
Epoch 341/1000, Loss: 0.008104857988655567
Epoch 351/1000, Loss: 0.00764047633856535
Epoch 361/1000, Loss: 0.00730994762852788
Epoch 371/1000, Loss: 0.01021517999470234
Epoch 381/1000, Loss: 0.006517296656966209
Epoch 391/1000, Loss: 0.006713935639709234
Epoch 401/1000, Loss: 0.00612868694588542
Epoch 411/1000, Loss: 0.006209659390151501
Epoch 421/1000, Loss: 0.0056778849102556705
Epoch 431/1000, Loss: 0.013621896505355835
Epoch 441/1000, Loss: 0.007102317176759243
Epoch 451/1000, Loss: 0.006090676877647638
Epoch 461/1000, Loss: 0.005737202242016792
Epoch 471/1000, Loss: 0.00559242628514

Cell 8.1：最终 Louvain compatibility audit

In [12]:
# ============================================================
# Cell 8.2
# FINAL no-training audit for RNA+ATAC Louvain backend
#
# Purpose:
# 1. Verify python-igraph Louvain responds to resolution
# 2. Verify deterministic output with fixed RNG
# 3. Verify saved E18.5 smoke partition is exactly reproducible
#
# NO training
# NO GT used for resolution selection
# NO ARI/NMI optimization
# ============================================================

from pathlib import Path
import random

import numpy as np
import scipy.sparse as sp
import anndata as ad
import scanpy as sc
import igraph as ig

from sklearn.metrics import adjusted_rand_score


SMOKE_DIR = Path(
    "/kaggle/working/COSMOS_baseline/smoke/E185_seed0"
)

embedding = np.load(
    SMOKE_DIR / "embedding.npy"
)

saved_pred = np.load(
    SMOKE_DIR / "pred_labels.npy",
    allow_pickle=True,
)


assert embedding.shape == (2129, 50)
assert np.isfinite(embedding).all()


# ============================================================
# 1. Build exactly the same 50-NN graph used by Cell 5
# ============================================================

adata = ad.AnnData(
    X=np.asarray(
        embedding,
        dtype=np.float32,
    )
)


sc.pp.neighbors(
    adata,
    n_neighbors=50,
    use_rep="X",
    random_state=0,
)


adjacency = (
    adata
    .obsp["connectivities"]
    .tocsr()
)


adjacency = adjacency.maximum(
    adjacency.T
)


upper = sp.triu(
    adjacency,
    k=1,
).tocoo()


edges = list(
    zip(
        upper.row.tolist(),
        upper.col.tolist(),
    )
)


graph = ig.Graph(
    n=adata.n_obs,
    edges=edges,
    directed=False,
)


print("=" * 110)
print("COSMOS DIRECT-IGRAPH LOUVAIN AUDIT")
print("=" * 110)

print(
    "Vertices:",
    graph.vcount()
)

print(
    "Edges   :",
    graph.ecount()
)


assert graph.vcount() == 2129
assert graph.ecount() > 0


# ============================================================
# 2. Deterministic runner
# ============================================================

def run_igraph_louvain(
    graph,
    resolution,
):

    ig.set_random_number_generator(
        random.Random(0)
    )


    partition = graph.community_multilevel(
        weights=None,
        return_levels=False,
        resolution=float(
            resolution
        ),
    )


    labels = np.asarray(
        partition.membership,
        dtype=str,
    )


    return labels


# ============================================================
# 3. Verify resolution really changes clustering
# ============================================================

test_resolutions = [
    0.50,
    1.00,
    1.40,
    1.66,
    2.00,
]


print(
    "\nResolution response:"
)


resolution_results = {}


for res in test_resolutions:

    pred = run_igraph_louvain(
        graph,
        res,
    )


    k = len(
        np.unique(pred)
    )


    resolution_results[
        res
    ] = pred


    print(
        f"resolution={res:.2f} "
        f"-> K={k}"
    )


cluster_counts = [
    len(
        np.unique(
            resolution_results[r]
        )
    )
    for r in test_resolutions
]


assert (
    len(
        set(
            cluster_counts
        )
    )
    > 1
), (
    "FAIL: resolution did not change "
    "the number of Louvain communities."
)


print(
    "\nPASS: direct igraph Louvain responds "
    "to resolution."
)


# ============================================================
# 4. Reproduce formal smoke resolution = 1.66 twice
# ============================================================

pred_a = run_igraph_louvain(
    graph,
    1.66,
)

pred_b = run_igraph_louvain(
    graph,
    1.66,
)


assert (
    len(
        np.unique(
            pred_a
        )
    )
    == 14
)


assert (
    len(
        np.unique(
            pred_b
        )
    )
    == 14
)


repeat_ari = adjusted_rand_score(
    pred_a,
    pred_b,
)


saved_ari = adjusted_rand_score(
    saved_pred,
    pred_a,
)


print(
    "\nDeterminism audit"
)

print(
    "K run A               :",
    len(
        np.unique(pred_a)
    )
)

print(
    "K run B               :",
    len(
        np.unique(pred_b)
    )
)

print(
    "ARI run A vs run B    :",
    repeat_ari
)

print(
    "ARI saved vs recompute:",
    saved_ari
)


assert np.isclose(
    repeat_ari,
    1.0,
    rtol=0,
    atol=1e-12,
)


assert np.isclose(
    saved_ari,
    1.0,
    rtol=0,
    atol=1e-12,
)


# ============================================================
# 5. Repeat full frozen resolution scan
# ============================================================

exact_hits = []


for res in np.arange(
    0.05,
    2.50 + 1e-12,
    0.01,
):

    res = float(
        np.round(
            res,
            8,
        )
    )


    pred = run_igraph_louvain(
        graph,
        res,
    )


    if (
        len(
            np.unique(pred)
        )
        == 14
    ):

        exact_hits.append(
            (
                res,
                pred.copy(),
            )
        )


assert len(
    exact_hits
) > 0


selected_res, selected_pred = (
    exact_hits[-1]
)


scan_ari = adjusted_rand_score(
    saved_pred,
    selected_pred,
)


print(
    "\nFull target-K scan audit"
)

print(
    "Exact-K resolutions:",
    len(exact_hits)
)

print(
    "Selected resolution :",
    selected_res
)

print(
    "Selected K          :",
    len(
        np.unique(
            selected_pred
        )
    )
)

print(
    "ARI saved vs scan   :",
    scan_ari
)


assert np.isclose(
    selected_res,
    1.66,
    rtol=0,
    atol=1e-12,
)


assert (
    len(
        np.unique(
            selected_pred
        )
    )
    == 14
)


assert np.isclose(
    scan_ari,
    1.0,
    rtol=0,
    atol=1e-12,
)


print(
    "\n" + "=" * 110
)

print(
    "COSMOS RNA+ATAC LOUVAIN BACKEND: PASS"
)

print(
    "=" * 110
)

print(
    "PASS: resolution affects Louvain partition."
)

print(
    "PASS: fixed RNG is deterministic."
)

print(
    "PASS: saved E18.5 smoke partition reproduced exactly."
)

print(
    "PASS: target-K scan reproduces resolution 1.66."
)

print(
    "\nFINAL DECISION:"
)

print(
    "Keep direct python-igraph community_multilevel "
    "for RNA+ATAC formal runs."
)

COSMOS DIRECT-IGRAPH LOUVAIN AUDIT
Vertices: 2129
Edges   : 69112

Resolution response:
resolution=0.50 -> K=5
resolution=1.00 -> K=10
resolution=1.40 -> K=13
resolution=1.66 -> K=14
resolution=2.00 -> K=15

PASS: direct igraph Louvain responds to resolution.

Determinism audit
K run A               : 14
K run B               : 14
ARI run A vs run B    : 1.0
ARI saved vs recompute: 1.0

Full target-K scan audit
Exact-K resolutions: 22
Selected resolution : 1.66
Selected K          : 14
ARI saved vs scan   : 1.0

COSMOS RNA+ATAC LOUVAIN BACKEND: PASS
PASS: resolution affects Louvain partition.
PASS: fixed RNG is deterministic.
PASS: saved E18.5 smoke partition reproduced exactly.
PASS: target-K scan reproduces resolution 1.66.

FINAL DECISION:
Keep direct python-igraph community_multilevel for RNA+ATAC formal runs.


Cell 9：COSMOS 正式 5 datasets × 10 seeds

In [14]:
# ============================================================
# Cell 9
# COSMOS FORMAL benchmark
#
# 5 datasets × seeds 0-9 = 50 formal runs
#
# IMPORTANT
# ----------
# 1. Smoke runs are NOT reused.
# 2. Formal runs are written to a separate directory.
# 3. Existing valid formal runs are skipped automatically.
# 4. GT / ARI / NMI are NEVER used for resolution selection.
# 5. Exact target K is required.
#
# Frozen downstream protocol:
#
# RNA+ADT:
#   COSMOS embedding
#   -> 50-NN graph
#   -> Leiden
#   -> resolution scan 0.05:0.01:2.50
#   -> LAST exact-target-K resolution
#
# RNA+ATAC:
#   COSMOS embedding
#   -> 50-NN graph
#   -> python-igraph Louvain community_multilevel
#   -> resolution scan 0.05:0.01:2.50
#   -> LAST exact-target-K resolution
#
# Evaluation:
#   ARI = sklearn adjusted_rand_score
#   NMI = normalized_mutual_info_score(..., average_method="max")
#   std = ddof=0
# ============================================================

from pathlib import Path
import json
import traceback
import time

import numpy as np
import pandas as pd


FORMAL_ROOT = Path(
    "/kaggle/working/COSMOS_baseline/formal_10seeds"
)

FORMAL_ROOT.mkdir(
    parents=True,
    exist_ok=True,
)


DATASET_ORDER = [
    "HLN-A1",
    "HLN-D1",
    "E18.5",
    "S2-E15",
    "S2-E18",
]


DATASET_FOLDER = {
    "HLN-A1": "HLNA1",
    "HLN-D1": "HLND1",
    "E18.5": "E185",
    "S2-E15": "S2E15",
    "S2-E18": "S2E18",
}


EXPECTED = {

    "HLN-A1": {
        "n_spots": 3484,
        "K": 10,
        "second_type": "ADT",
        "cluster_method": "leiden",
        "spatial_reg": 0.05,
    },

    "HLN-D1": {
        "n_spots": 3359,
        "K": 11,
        "second_type": "ADT",
        "cluster_method": "leiden",
        "spatial_reg": 0.05,
    },

    "E18.5": {
        "n_spots": 2129,
        "K": 14,
        "second_type": "ATAC",
        "cluster_method": "louvain",
        "spatial_reg": 0.01,
    },

    "S2-E15": {
        "n_spots": 1939,
        "K": 15,
        "second_type": "ATAC",
        "cluster_method": "louvain",
        "spatial_reg": 0.01,
    },

    "S2-E18": {
        "n_spots": 2248,
        "K": 16,
        "second_type": "ATAC",
        "cluster_method": "louvain",
        "spatial_reg": 0.01,
    },
}


# ============================================================
# 1. Freeze machine-readable protocol
# ============================================================

FORMAL_PROTOCOL = {

    "method":
        "COSMOS",

    "experiment":
        "5 datasets x 10 training seeds",

    "datasets":
        DATASET_ORDER,

    "training_seeds":
        list(range(10)),

    "smoke_runs_included_in_formal":
        False,

    "benchmark_population":
        "same frozen benchmark spots as SpaMGCL",

    "training": {

        "z_dim":
            50,

        "lr":
            0.001,

        "wnn_epoch":
            500,

        "total_epoch":
            1000,

        "max_patience_bef":
            10,

        "max_patience_aft":
            30,

        "min_stop":
            200,

        "spatial_neighbors":
            10,

        "regularization_acceleration":
            True,

        "edge_subset_sz":
            1000000,
    },

    "modality_specific": {

        "RNA+ADT": {

            "RNA_preprocessing":
                "normalize_per_cell + log1p",

            "ADT_preprocessing":
                "log1p",

            "spatial_regularization_strength":
                0.05,

            "clustering":
                "Leiden via Scanpy/leidenalg",
        },

        "RNA+ATAC": {

            "additional_preprocessing_before_COSMOS":
                False,

            "spatial_regularization_strength":
                0.01,

            "clustering":
                (
                    "Louvain via python-igraph "
                    "community_multilevel"
                ),

            "compatibility_reason":
                (
                    "legacy PyPI louvain backend is not "
                    "compatible with Python 3.12; "
                    "Louvain algorithm retained through "
                    "python-igraph"
                ),
        },
    },

    "clustering_protocol": {

        "n_neighbors":
            50,

        "resolution_start":
            0.05,

        "resolution_end":
            2.50,

        "resolution_step":
            0.01,

        "selection_rule":
            (
                "scan upward and retain the LAST "
                "resolution producing exact target K"
            ),

        "selection_uses_ground_truth":
            False,

        "selection_uses_ARI_or_NMI":
            False,

        "cluster_random_state":
            0,

        "exact_target_K_required":
            True,
    },

    "evaluation": {

        "ARI":
            "sklearn adjusted_rand_score",

        "NMI":
            (
                "sklearn normalized_mutual_info_score "
                "average_method=max"
            ),

        "std_ddof":
            0,
    },

    "environment": {

        "python":
            "3.12.13",

        "torch":
            "2.10.0+cu128",

        "numpy":
            "2.0.2",

        "scipy":
            "1.16.3",

        "scanpy":
            "1.11.4",

        "anndata":
            "0.11.4",

        "sklearn":
            "1.6.1",

        "torch_geometric":
            "2.8.0.post1",

        "igraph":
            "1.0.0",

        "leidenalg":
            "0.12.0",

        "gpu":
            "Tesla T4",
    },

    "COSMOS_git_commit":
        COSMOS_COMMIT,

    "RNA_ATAC_backend_audit": {

        "dataset":
            "E18.5 seed0 smoke",

        "vertices":
            2129,

        "edges":
            69112,

        "resolution_response_verified":
            True,

        "deterministic_fixed_rng":
            True,

        "saved_partition_reproduced":
            True,

        "smoke_selected_resolution":
            1.66,

        "smoke_exact_K_resolutions":
            22,
    },
}


PROTOCOL_PATH = (
    FORMAL_ROOT
    / "protocol.json"
)


with PROTOCOL_PATH.open(
    "w",
    encoding="utf-8",
) as f:

    json.dump(
        FORMAL_PROTOCOL,
        f,
        indent=2,
    )


print("=" * 110)
print("COSMOS FORMAL PROTOCOL FROZEN")
print("=" * 110)

print(
    "Protocol:",
    PROTOCOL_PATH
)

print(
    "COSMOS commit:",
    COSMOS_COMMIT
)


# ============================================================
# 2. Existing-run validator
# ============================================================

REQUIRED_FILES = [
    "metrics.json",
    "embedding.npy",
    "modality_weights.npy",
    "pred_labels.npy",
    "gt_labels.npy",
    "coords.npy",
    "spot_ids.npy",
]


def formal_run_dir(
    dataset,
    seed,
):

    return (
        FORMAL_ROOT
        / (
            DATASET_FOLDER[
                dataset
            ]
            + f"_seed{seed}"
        )
    )


def valid_existing_run(
    dataset,
    seed,
    verbose=False,
):

    run_dir = formal_run_dir(
        dataset,
        seed,
    )


    expected = EXPECTED[
        dataset
    ]


    # --------------------------------------------------------
    # Required files
    # --------------------------------------------------------

    for name in REQUIRED_FILES:

        path = (
            run_dir
            / name
        )

        if not path.exists():

            if verbose:
                print(
                    "Missing:",
                    path
                )

            return False


    try:

        # ----------------------------------------------------
        # Metadata
        # ----------------------------------------------------

        with (
            run_dir
            / "metrics.json"
        ).open(
            "r",
            encoding="utf-8",
        ) as f:

            m = json.load(f)


        if (
            m["dataset"]
            != dataset
        ):

            return False


        if (
            int(
                m["training_seed"]
            )
            != seed
        ):

            return False


        if (
            int(
                m["n_spots"]
            )
            != expected[
                "n_spots"
            ]
        ):

            return False


        if (
            int(
                m["target_K"]
            )
            != expected[
                "K"
            ]
        ):

            return False


        if (
            int(
                m["predicted_K"]
            )
            != expected[
                "K"
            ]
        ):

            return False


        if (
            m[
                "cluster_method"
            ]
            != expected[
                "cluster_method"
            ]
        ):

            return False


        if not np.isclose(
            float(
                m[
                    "spatial_regularization_strength"
                ]
            ),
            expected[
                "spatial_reg"
            ],
            rtol=0,
            atol=1e-12,
        ):

            return False


        if not np.isfinite(
            float(
                m["ARI"]
            )
        ):

            return False


        if not np.isfinite(
            float(
                m["NMI"]
            )
        ):

            return False


        # ----------------------------------------------------
        # Evidence arrays
        # ----------------------------------------------------

        embedding = np.load(
            run_dir
            / "embedding.npy"
        )

        weights = np.load(
            run_dir
            / "modality_weights.npy"
        )

        pred = np.load(
            run_dir
            / "pred_labels.npy",
            allow_pickle=True,
        )

        gt = np.load(
            run_dir
            / "gt_labels.npy",
            allow_pickle=True,
        )

        coords = np.load(
            run_dir
            / "coords.npy"
        )

        spot_ids = np.load(
            run_dir
            / "spot_ids.npy",
            allow_pickle=True,
        )


        n = expected[
            "n_spots"
        ]


        if (
            embedding.shape
            != (n, 50)
        ):

            return False


        if (
            weights.shape
            != (n, 2)
        ):

            return False


        if (
            len(pred)
            != n
        ):

            return False


        if (
            len(gt)
            != n
        ):

            return False


        if (
            coords.shape
            != (n, 2)
        ):

            return False


        if (
            len(spot_ids)
            != n
        ):

            return False


        if not np.isfinite(
            embedding
        ).all():

            return False


        if not np.isfinite(
            weights
        ).all():

            return False


        if not np.isfinite(
            coords
        ).all():

            return False


        if (
            len(
                np.unique(
                    pred
                )
            )
            != expected[
                "K"
            ]
        ):

            return False


        if (
            len(
                np.unique(
                    gt
                )
            )
            != expected[
                "K"
            ]
        ):

            return False


        if (
            len(
                np.unique(
                    spot_ids
                )
            )
            != n
        ):

            return False


        # COSMOS WNN weights should sum to ~1 per spot.
        if not np.allclose(
            weights.sum(
                axis=1
            ),
            1.0,
            rtol=0,
            atol=1e-5,
        ):

            return False


        return True


    except Exception as e:

        if verbose:

            print(
                "Validation exception:",
                repr(e)
            )

        return False


# ============================================================
# 3. Count already-valid runs
# ============================================================

existing_valid = []


for dataset in DATASET_ORDER:

    for seed in range(10):

        if valid_existing_run(
            dataset,
            seed,
        ):

            existing_valid.append(
                (
                    dataset,
                    seed,
                )
            )


print(
    "\nExisting valid formal runs:",
    len(existing_valid),
    "/ 50"
)


if existing_valid:

    print(
        existing_valid
    )


# ============================================================
# 4. Formal 50-run loop
# ============================================================

formal_start = time.time()


for dataset in DATASET_ORDER:

    print(
        "\n\n" + "#" * 110
    )

    print(
        f"# FORMAL DATASET: {dataset}"
    )

    print(
        "#" * 110
    )


    for seed in range(10):

        run_dir = formal_run_dir(
            dataset,
            seed,
        )


        # ----------------------------------------------------
        # Resume support
        # ----------------------------------------------------

        if valid_existing_run(
            dataset,
            seed,
        ):

            print(
                f"\n[SKIP] "
                f"{dataset} seed={seed} "
                f"already valid."
            )

            continue


        print(
            "\n" + "=" * 110
        )

        print(
            f"FORMAL | COSMOS | "
            f"{dataset} | seed={seed}"
        )

        print(
            "=" * 110
        )


        # Remove old error marker before retry.
        error_path = (
            run_dir
            / "ERROR.json"
        )

        if error_path.exists():

            error_path.unlink()


        try:

            # =================================================
            # Run exactly the frozen single-run protocol
            # =================================================

            result = run_cosmos_once(

                dataset_name=dataset,

                seed=seed,

                stage="formal_10seeds",
            )


            # =================================================
            # Immediately validate saved evidence
            # =================================================

            assert valid_existing_run(
                dataset,
                seed,
                verbose=True,
            ), (
                f"Saved run failed audit: "
                f"{dataset} seed={seed}"
            )


            print(
                f"\nFORMAL PASS | "
                f"{dataset} | "
                f"seed={seed} | "
                f"ARI={result['ARI']:.6f} | "
                f"NMI={result['NMI']:.6f} | "
                f"K={result['predicted_K']}"
            )


        except Exception as e:

            run_dir.mkdir(
                parents=True,
                exist_ok=True,
            )


            error_record = {

                "dataset":
                    dataset,

                "seed":
                    int(seed),

                "error_type":
                    type(e).__name__,

                "error":
                    str(e),

                "traceback":
                    traceback.format_exc(),
            }


            with error_path.open(
                "w",
                encoding="utf-8",
            ) as f:

                json.dump(
                    error_record,
                    f,
                    indent=2,
                )


            print(
                f"\nFAILED | "
                f"{dataset} | seed={seed}"
            )

            print(
                repr(e)
            )

            print(
                "Error record:",
                error_path
            )


# ============================================================
# 5. Reconstruct RAW from disk
# ============================================================

rows = []


for dataset in DATASET_ORDER:

    for seed in range(10):

        if not valid_existing_run(
            dataset,
            seed,
        ):

            continue


        run_dir = formal_run_dir(
            dataset,
            seed,
        )


        with (
            run_dir
            / "metrics.json"
        ).open(
            "r",
            encoding="utf-8",
        ) as f:

            m = json.load(f)


        rows.append(
            m
        )


raw_df = pd.DataFrame(
    rows
)


if len(raw_df) > 0:

    raw_df = (
        raw_df
        .sort_values(
            [
                "dataset",
                "training_seed",
            ]
        )
        .reset_index(
            drop=True
        )
    )


RAW_PATH = (
    FORMAL_ROOT
    / "COSMOS_5datasets_10seeds_RAW.csv"
)


raw_df.to_csv(
    RAW_PATH,
    index=False,
)


# ============================================================
# 6. Completion report
# ============================================================

print(
    "\n\n" + "=" * 110
)

print(
    "COSMOS FORMAL RUN STATUS"
)

print(
    "=" * 110
)


counts = {}


for dataset in DATASET_ORDER:

    count = 0


    for seed in range(10):

        if valid_existing_run(
            dataset,
            seed,
        ):

            count += 1


    counts[
        dataset
    ] = count


    print(
        f"{dataset:8s}: "
        f"{count}/10 valid runs"
    )


total_valid = sum(
    counts.values()
)


print(
    "\nTOTAL:",
    f"{total_valid}/50"
)


print(
    "Elapsed:",
    f"{(time.time() - formal_start) / 60:.2f} min"
)


print(
    "\nRAW:",
    RAW_PATH
)


# ============================================================
# 7. Build SUMMARY only when all 50 are complete
# ============================================================

if total_valid == 50:

    summary_rows = []


    for dataset in DATASET_ORDER:

        part = (
            raw_df[
                raw_df[
                    "dataset"
                ]
                == dataset
            ]
            .sort_values(
                "training_seed"
            )
        )


        assert len(
            part
        ) == 10


        assert (
            sorted(
                part[
                    "training_seed"
                ]
                .astype(int)
                .tolist()
            )
            == list(
                range(10)
            )
        )


        summary_rows.append(
            {

                "dataset":
                    dataset,

                "n_runs":
                    10,

                "ARI_mean":
                    float(
                        part[
                            "ARI"
                        ].mean()
                    ),

                "ARI_std":
                    float(
                        part[
                            "ARI"
                        ].std(
                            ddof=0
                        )
                    ),

                "NMI_mean":
                    float(
                        part[
                            "NMI"
                        ].mean()
                    ),

                "NMI_std":
                    float(
                        part[
                            "NMI"
                        ].std(
                            ddof=0
                        )
                    ),

                "exact_K_runs":
                    int(
                        (
                            part[
                                "predicted_K"
                            ].astype(int)
                            ==
                            EXPECTED[
                                dataset
                            ][
                                "K"
                            ]
                        ).sum()
                    ),
            }
        )


    summary_df = pd.DataFrame(
        summary_rows
    )


    SUMMARY_PATH = (
        FORMAL_ROOT
        / "COSMOS_5datasets_10seeds_SUMMARY.csv"
    )


    summary_df.to_csv(
        SUMMARY_PATH,
        index=False,
    )


    print(
        "\n" + "=" * 110
    )

    print(
        "COSMOS FORMAL SUMMARY"
    )

    print(
        "=" * 110
    )


    for _, row in (
        summary_df.iterrows()
    ):

        print(
            f"{row['dataset']:8s} | "
            f"ARI "
            f"{row['ARI_mean']:.6f}"
            f" ± "
            f"{row['ARI_std']:.6f}"
            f" | NMI "
            f"{row['NMI_mean']:.6f}"
            f" ± "
            f"{row['NMI_std']:.6f}"
            f" | exact-K "
            f"{int(row['exact_K_runs'])}/10"
        )


    print(
        "\nSUMMARY:",
        SUMMARY_PATH
    )


    print(
        "\nPASS: 50/50 COSMOS formal runs complete."
    )

    print(
        "PASS: seeds 0-9 complete for all datasets."
    )

    print(
        "PASS: all runs reached exact target K."
    )

    print(
        "PASS: std uses ddof=0."
    )


else:

    print(
        "\nNOT COMPLETE YET."
    )

    print(
        "Re-run this same Cell 9 later; "
        "valid runs will be skipped automatically."
    )

COSMOS FORMAL PROTOCOL FROZEN
Protocol: /kaggle/working/COSMOS_baseline/formal_10seeds/protocol.json
COSMOS commit: 56ea355be51e64d9253e2871b8bd447fdfd0d230

Existing valid formal runs: 0 / 50


##############################################################################################################
# FORMAL DATASET: HLN-A1
##############################################################################################################

FORMAL | COSMOS | HLN-A1 | seed=0


/usr/local/lib/python3.12/dist-packages/anndata/_core/anndata.py:1758: UserWarning: Variable names are not unique. To make them unique, call `.var_names_make_unique`.
  utils.warn_names_duplicates("var")
/usr/local/lib/python3.12/dist-packages/anndata/_core/anndata.py:1758: UserWarning: Variable names are not unique. To make them unique, call `.var_names_make_unique`.
  utils.warn_names_duplicates("var")
/tmp/ipykernel_58/2485005381.py:273: FutureWarning: Use sc.pp.normalize_total instead
  sc.pp.normalize_per_cell(
/usr/local/lib/python3.12/dist-packages/scanpy/preprocessing/_simple.py:591: FutureWarning: Use sc.pp.normalize_total instead
  normalize_per_cell(


HLN-A1: 3484 spots | K=10 | RNA + ADT | spatial_reg=0.05 | leiden

COSMOS | HLN-A1 | seed=0
Input spots : 3484
Target K    : 10
Modality    : RNA + ADT
Cluster     : leiden
Spatial reg : 0.05
Epoch 1/1000, Loss: 1.4408732652664185
Epoch 11/1000, Loss: 1.4229328632354736
Epoch 21/1000, Loss: 1.416532278060913
Epoch 31/1000, Loss: 1.4125770330429077
Epoch 41/1000, Loss: 1.4097596406936646
Epoch 51/1000, Loss: 1.4075915813446045
Epoch 61/1000, Loss: 1.4039438962936401
Epoch 71/1000, Loss: 1.3976424932479858
Epoch 81/1000, Loss: 1.385579228401184
Epoch 91/1000, Loss: 1.3390567302703857
Epoch 101/1000, Loss: 1.1893527507781982
Epoch 111/1000, Loss: 0.8017709851264954
Epoch 121/1000, Loss: 0.4951621890068054
Epoch 131/1000, Loss: 0.39597490429878235
Epoch 141/1000, Loss: 0.2143106758594513
Epoch 151/1000, Loss: 0.11788971722126007
Epoch 161/1000, Loss: 0.08342504501342773
Epoch 171/1000, Loss: 0.059403713792562485
Epoch 181/1000, Loss: 0.05285075306892395
Epoch 191/1000, Loss: 0.050659593194

/usr/local/lib/python3.12/dist-packages/anndata/_core/anndata.py:1758: UserWarning: Variable names are not unique. To make them unique, call `.var_names_make_unique`.
  utils.warn_names_duplicates("var")
/usr/local/lib/python3.12/dist-packages/anndata/_core/anndata.py:1758: UserWarning: Variable names are not unique. To make them unique, call `.var_names_make_unique`.
  utils.warn_names_duplicates("var")
/tmp/ipykernel_58/2485005381.py:273: FutureWarning: Use sc.pp.normalize_total instead
  sc.pp.normalize_per_cell(
/usr/local/lib/python3.12/dist-packages/scanpy/preprocessing/_simple.py:591: FutureWarning: Use sc.pp.normalize_total instead
  normalize_per_cell(


HLN-A1: 3484 spots | K=10 | RNA + ADT | spatial_reg=0.05 | leiden

COSMOS | HLN-A1 | seed=1
Input spots : 3484
Target K    : 10
Modality    : RNA + ADT
Cluster     : leiden
Spatial reg : 0.05
Epoch 1/1000, Loss: 1.4204604625701904
Epoch 11/1000, Loss: 1.4206976890563965
Epoch 21/1000, Loss: 1.41350519657135
Epoch 31/1000, Loss: 1.4080668687820435
Epoch 41/1000, Loss: 1.4008731842041016
Epoch 51/1000, Loss: 1.3887487649917603
Epoch 61/1000, Loss: 1.3546667098999023
Epoch 71/1000, Loss: 1.2123072147369385
Epoch 81/1000, Loss: 0.9739124774932861
Epoch 91/1000, Loss: 0.8844431042671204
Epoch 101/1000, Loss: 0.6388108134269714
Epoch 111/1000, Loss: 0.4368366599082947
Epoch 121/1000, Loss: 0.2571581304073334
Epoch 131/1000, Loss: 0.14321282505989075
Epoch 141/1000, Loss: 0.10719097405672073
Epoch 151/1000, Loss: 0.0866975411772728
Epoch 161/1000, Loss: 0.05880122631788254
Epoch 171/1000, Loss: 0.050861284136772156
Epoch 181/1000, Loss: 0.05124834179878235
Epoch 191/1000, Loss: 0.050742864608

...

S2-E18: 2248 spots | K=16 | RNA + ATAC | spatial_reg=0.01 | louvain

COSMOS | S2-E18 | seed=7
Input spots : 2248
Target K    : 16
Modality    : RNA + ATAC
Cluster     : louvain
Spatial reg : 0.01
Epoch 1/1000, Loss: 1.4016042947769165
Epoch 11/1000, Loss: 1.39797842502594
Epoch 21/1000, Loss: 1.3918554782867432
Epoch 31/1000, Loss: 1.388728141784668
Epoch 41/1000, Loss: 1.3842988014221191
Epoch 51/1000, Loss: 1.3773082494735718
Epoch 61/1000, Loss: 1.3607453107833862
Epoch 71/1000, Loss: 1.3048397302627563
Epoch 81/1000, Loss: 1.1379423141479492
Epoch 91/1000, Loss: 0.8162204623222351
Epoch 101/1000, Loss: 0.49023205041885376
Epoch 111/1000, Loss: 0.26047781109809875
Epoch 121/1000, Loss: 0.11347696185112
Epoch 131/1000, Loss: 0.061318036168813705
Epoch 141/1000, Loss: 0.04034511744976044
Epoch 151/1000, Loss: 0.025311468169093132
Epoch 161/1000, Loss: 0.022136442363262177
Epoch 171/1000, Loss: 0.01674679107964039
Epoch 181/1000, Loss: 0.011049002408981323
Epoch 191/1000, Loss: 0.01031

/usr/local/lib/python3.12/dist-packages/anndata/_core/anndata.py:1758: UserWarning: Variable names are not unique. To make them unique, call `.var_names_make_unique`.
  utils.warn_names_duplicates("var")


S2-E18: 2248 spots | K=16 | RNA + ATAC | spatial_reg=0.01 | louvain

COSMOS | S2-E18 | seed=8
Input spots : 2248
Target K    : 16
Modality    : RNA + ATAC
Cluster     : louvain
Spatial reg : 0.01
Epoch 1/1000, Loss: 1.4104714393615723
Epoch 11/1000, Loss: 1.3940025568008423
Epoch 21/1000, Loss: 1.3905402421951294
Epoch 31/1000, Loss: 1.3893979787826538
Epoch 41/1000, Loss: 1.3877066373825073
Epoch 51/1000, Loss: 1.3854994773864746
Epoch 61/1000, Loss: 1.3804633617401123
Epoch 71/1000, Loss: 1.3633970022201538
Epoch 81/1000, Loss: 1.242753267288208
Epoch 91/1000, Loss: 0.9658530950546265
Epoch 101/1000, Loss: 0.6125729084014893
Epoch 111/1000, Loss: 0.3292604982852936
Epoch 121/1000, Loss: 0.15108992159366608
Epoch 131/1000, Loss: 0.07353673130273819
Epoch 141/1000, Loss: 0.03832778334617615
Epoch 151/1000, Loss: 0.025186413899064064
Epoch 161/1000, Loss: 0.020359188318252563
Epoch 171/1000, Loss: 0.016363035887479782
Epoch 181/1000, Loss: 0.017652036622166634
Epoch 191/1000, Loss: 0.01

/usr/local/lib/python3.12/dist-packages/anndata/_core/anndata.py:1758: UserWarning: Variable names are not unique. To make them unique, call `.var_names_make_unique`.
  utils.warn_names_duplicates("var")


S2-E18: 2248 spots | K=16 | RNA + ATAC | spatial_reg=0.01 | louvain

COSMOS | S2-E18 | seed=9
Input spots : 2248
Target K    : 16
Modality    : RNA + ATAC
Cluster     : louvain
Spatial reg : 0.01
Epoch 1/1000, Loss: 1.3953237533569336
Epoch 11/1000, Loss: 1.4012702703475952
Epoch 21/1000, Loss: 1.3917804956436157
Epoch 31/1000, Loss: 1.386609673500061
Epoch 41/1000, Loss: 1.3828006982803345
Epoch 51/1000, Loss: 1.3772037029266357
Epoch 61/1000, Loss: 1.3651939630508423
Epoch 71/1000, Loss: 1.3315414190292358
Epoch 81/1000, Loss: 1.154269814491272
Epoch 91/1000, Loss: 0.7749012112617493
Epoch 101/1000, Loss: 0.4258427619934082
Epoch 111/1000, Loss: 0.22541721165180206
Epoch 121/1000, Loss: 0.09270726144313812
Epoch 131/1000, Loss: 0.05751381441950798
Epoch 141/1000, Loss: 0.037127383053302765
Epoch 151/1000, Loss: 0.02822667360305786
Epoch 161/1000, Loss: 0.022428477182984352
Epoch 171/1000, Loss: 0.014689171686768532
Epoch 181/1000, Loss: 0.015966810286045074
Epoch 191/1000, Loss: 0.01

Cell 10：COSMOS 独立审计

In [15]:
# ============================================================
# Cell 10
# COSMOS FINAL independent audit
# ============================================================

from pathlib import Path
import json

import numpy as np
import pandas as pd

from sklearn.metrics import (
    adjusted_rand_score,
    normalized_mutual_info_score,
)


FORMAL_ROOT = Path(
    "/kaggle/working/COSMOS_baseline/formal_10seeds"
)


DATASET_ORDER = [
    "HLN-A1",
    "HLN-D1",
    "E18.5",
    "S2-E15",
    "S2-E18",
]


DATASET_FOLDER = {
    "HLN-A1": "HLNA1",
    "HLN-D1": "HLND1",
    "E18.5": "E185",
    "S2-E15": "S2E15",
    "S2-E18": "S2E18",
}


EXPECTED = {
    "HLN-A1": {"n_spots": 3484, "K": 10},
    "HLN-D1": {"n_spots": 3359, "K": 11},
    "E18.5": {"n_spots": 2129, "K": 14},
    "S2-E15": {"n_spots": 1939, "K": 15},
    "S2-E18": {"n_spots": 2248, "K": 16},
}


RAW_PATH = (
    FORMAL_ROOT
    / "COSMOS_5datasets_10seeds_RAW.csv"
)

SUMMARY_PATH = (
    FORMAL_ROOT
    / "COSMOS_5datasets_10seeds_SUMMARY.csv"
)

PROTOCOL_PATH = (
    FORMAL_ROOT
    / "protocol.json"
)


assert RAW_PATH.exists()
assert SUMMARY_PATH.exists()
assert PROTOCOL_PATH.exists()


raw_saved = pd.read_csv(RAW_PATH)
summary_saved = pd.read_csv(SUMMARY_PATH)

audit_rows = []


print("=" * 110)
print("COSMOS FINAL 50-RUN INDEPENDENT AUDIT")
print("=" * 110)


for dataset in DATASET_ORDER:

    print(f"\nAuditing {dataset}...")

    n = EXPECTED[dataset]["n_spots"]
    K = EXPECTED[dataset]["K"]


    for seed in range(10):

        run_dir = (
            FORMAL_ROOT
            / f"{DATASET_FOLDER[dataset]}_seed{seed}"
        )


        required = {
            "metrics": run_dir / "metrics.json",
            "embedding": run_dir / "embedding.npy",
            "weights": run_dir / "modality_weights.npy",
            "pred": run_dir / "pred_labels.npy",
            "gt": run_dir / "gt_labels.npy",
            "coords": run_dir / "coords.npy",
            "spot_ids": run_dir / "spot_ids.npy",
        }


        for name, path in required.items():
            assert path.exists(), (
                f"{dataset} seed={seed}: "
                f"missing {name}"
            )


        with required["metrics"].open(
            "r",
            encoding="utf-8",
        ) as f:

            m = json.load(f)


        embedding = np.load(
            required["embedding"]
        )

        weights = np.load(
            required["weights"]
        )

        pred = np.load(
            required["pred"],
            allow_pickle=True,
        )

        gt = np.load(
            required["gt"],
            allow_pickle=True,
        )

        coords = np.load(
            required["coords"]
        )

        spot_ids = np.load(
            required["spot_ids"],
            allow_pickle=True,
        )


        # ----------------------------------------------------
        # Metadata
        # ----------------------------------------------------

        assert m["dataset"] == dataset
        assert int(m["training_seed"]) == seed
        assert int(m["n_spots"]) == n
        assert int(m["target_K"]) == K
        assert int(m["predicted_K"]) == K


        # ----------------------------------------------------
        # Shapes / finite values
        # ----------------------------------------------------

        assert embedding.shape == (n, 50)
        assert weights.shape == (n, 2)
        assert coords.shape == (n, 2)

        assert len(pred) == n
        assert len(gt) == n
        assert len(spot_ids) == n

        assert np.isfinite(embedding).all()
        assert np.isfinite(weights).all()
        assert np.isfinite(coords).all()

        assert len(np.unique(spot_ids)) == n


        # ----------------------------------------------------
        # WNN weights
        # ----------------------------------------------------

        assert np.allclose(
            weights.sum(axis=1),
            1.0,
            rtol=0,
            atol=1e-5,
        )


        # ----------------------------------------------------
        # K
        # ----------------------------------------------------

        assert len(np.unique(gt)) == K
        assert len(np.unique(pred)) == K


        # ----------------------------------------------------
        # Independent metric recomputation
        # ----------------------------------------------------

        ari = adjusted_rand_score(
            gt,
            pred,
        )

        nmi = normalized_mutual_info_score(
            gt,
            pred,
            average_method="max",
        )


        assert np.isclose(
            ari,
            float(m["ARI"]),
            rtol=0,
            atol=1e-12,
        )


        assert np.isclose(
            nmi,
            float(m["NMI"]),
            rtol=0,
            atol=1e-12,
        )


        audit_rows.append({
            "dataset": dataset,
            "training_seed": seed,
            "n_spots": n,
            "target_K": K,
            "predicted_K": len(np.unique(pred)),
            "ARI": float(ari),
            "NMI": float(nmi),
        })


    print(
        f"{dataset}: 10/10 PASS"
    )


# ============================================================
# RAW audit
# ============================================================

audit_raw = pd.DataFrame(
    audit_rows
)

assert len(audit_raw) == 50


left = (
    audit_raw[
        [
            "dataset",
            "training_seed",
            "ARI",
            "NMI",
        ]
    ]
    .sort_values(
        ["dataset", "training_seed"]
    )
    .reset_index(drop=True)
)


right = (
    raw_saved[
        [
            "dataset",
            "training_seed",
            "ARI",
            "NMI",
        ]
    ]
    .sort_values(
        ["dataset", "training_seed"]
    )
    .reset_index(drop=True)
)


assert left[
    ["dataset", "training_seed"]
].equals(
    right[
        ["dataset", "training_seed"]
    ]
)


assert np.allclose(
    left["ARI"],
    right["ARI"],
    rtol=0,
    atol=1e-12,
)


assert np.allclose(
    left["NMI"],
    right["NMI"],
    rtol=0,
    atol=1e-12,
)


print(
    "\nPASS: RAW.csv verified."
)


# ============================================================
# SUMMARY audit
# ============================================================

summary_rows = []


for dataset in DATASET_ORDER:

    part = audit_raw[
        audit_raw["dataset"] == dataset
    ]


    summary_rows.append({

        "dataset":
            dataset,

        "n_runs":
            10,

        "ARI_mean":
            float(
                part["ARI"].mean()
            ),

        "ARI_std":
            float(
                part["ARI"].std(
                    ddof=0
                )
            ),

        "NMI_mean":
            float(
                part["NMI"].mean()
            ),

        "NMI_std":
            float(
                part["NMI"].std(
                    ddof=0
                )
            ),

        "exact_K_runs":
            int(
                (
                    part["predicted_K"]
                    ==
                    part["target_K"]
                ).sum()
            ),
    })


audit_summary = pd.DataFrame(
    summary_rows
)


saved_summary = (
    summary_saved
    .set_index("dataset")
    .loc[DATASET_ORDER]
    .reset_index()
)


for col in [
    "ARI_mean",
    "ARI_std",
    "NMI_mean",
    "NMI_std",
]:

    assert np.allclose(
        audit_summary[col],
        saved_summary[col],
        rtol=0,
        atol=1e-12,
    )


assert (
    audit_summary[
        "exact_K_runs"
    ] == 10
).all()


print(
    "PASS: SUMMARY.csv verified."
)


# ============================================================
# Save audit files
# ============================================================

AUDIT_RAW_PATH = (
    FORMAL_ROOT
    / "COSMOS_5datasets_10seeds_AUDIT_RAW.csv"
)

AUDIT_SUMMARY_PATH = (
    FORMAL_ROOT
    / "COSMOS_5datasets_10seeds_AUDIT_SUMMARY.csv"
)


audit_raw.to_csv(
    AUDIT_RAW_PATH,
    index=False,
)

audit_summary.to_csv(
    AUDIT_SUMMARY_PATH,
    index=False,
)


print(
    "\n" + "=" * 110
)

print(
    "COSMOS INDEPENDENT SUMMARY"
)

print(
    "=" * 110
)


for _, row in audit_summary.iterrows():

    print(
        f"{row['dataset']:8s} | "
        f"ARI "
        f"{row['ARI_mean']:.6f}"
        f" ± "
        f"{row['ARI_std']:.6f}"
        f" | NMI "
        f"{row['NMI_mean']:.6f}"
        f" ± "
        f"{row['NMI_std']:.6f}"
        f" | exact-K "
        f"{int(row['exact_K_runs'])}/10"
    )


print(
    "\nPASS: 50/50 COSMOS runs independently verified."
)

print(
    "PASS: metrics.json == recomputed metrics."
)

print(
    "PASS: RAW == independent recomputation."
)

print(
    "PASS: SUMMARY == independent recomputation."
)

print(
    "PASS: all 50 runs have exact target K."
)

print(
    "PASS: WNN modality weights verified."
)

print(
    "PASS: std uses ddof=0."
)

COSMOS FINAL 50-RUN INDEPENDENT AUDIT

Auditing HLN-A1...
HLN-A1: 10/10 PASS

Auditing HLN-D1...
HLN-D1: 10/10 PASS

Auditing E18.5...
E18.5: 10/10 PASS

Auditing S2-E15...
S2-E15: 10/10 PASS

Auditing S2-E18...
S2-E18: 10/10 PASS

PASS: RAW.csv verified.
PASS: SUMMARY.csv verified.

COSMOS INDEPENDENT SUMMARY
HLN-A1   | ARI 0.139880 ± 0.039926 | NMI 0.243262 ± 0.055027 | exact-K 10/10
HLN-D1   | ARI 0.131119 ± 0.030523 | NMI 0.232858 ± 0.032248 | exact-K 10/10
E18.5    | ARI 0.205687 ± 0.017547 | NMI 0.378364 ± 0.024838 | exact-K 10/10
S2-E15   | ARI 0.238658 ± 0.021561 | NMI 0.435317 ± 0.018831 | exact-K 10/10
S2-E18   | ARI 0.216324 ± 0.025496 | NMI 0.371493 ± 0.031020 | exact-K 10/10

PASS: 50/50 COSMOS runs independently verified.
PASS: metrics.json == recomputed metrics.
PASS: RAW == independent recomputation.
PASS: SUMMARY == independent recomputation.
PASS: all 50 runs have exact target K.
PASS: WNN modality weights verified.
PASS: std uses ddof=0.


Cell 11：COSMOS 最终打包

In [16]:
# ============================================================
# Cell 11
# COSMOS FINAL archive
# ============================================================

from pathlib import Path
import shutil
import hashlib


FORMAL_ROOT = Path(
    "/kaggle/working/COSMOS_baseline/formal_10seeds"
)


ARCHIVE_NAME = (
    "COSMOS_5datasets_10seeds_FORMAL_FINAL"
)


STAGE_DIR = (
    Path("/kaggle/working")
    / ARCHIVE_NAME
)


ZIP_BASE = (
    Path("/kaggle/working")
    / ARCHIVE_NAME
)


ZIP_PATH = Path(
    str(ZIP_BASE) + ".zip"
)


if STAGE_DIR.exists():
    shutil.rmtree(STAGE_DIR)

if ZIP_PATH.exists():
    ZIP_PATH.unlink()


STAGE_DIR.mkdir(
    parents=True,
    exist_ok=True,
)


# ------------------------------------------------------------
# Top-level evidence
# ------------------------------------------------------------

TOP_FILES = [

    "protocol.json",

    "COSMOS_5datasets_10seeds_RAW.csv",

    "COSMOS_5datasets_10seeds_SUMMARY.csv",

    "COSMOS_5datasets_10seeds_AUDIT_RAW.csv",

    "COSMOS_5datasets_10seeds_AUDIT_SUMMARY.csv",
]


for name in TOP_FILES:

    src = (
        FORMAL_ROOT / name
    )

    assert src.exists(), src

    shutil.copy2(
        src,
        STAGE_DIR / name,
    )


DATASET_FOLDER = {
    "HLN-A1": "HLNA1",
    "HLN-D1": "HLND1",
    "E18.5": "E185",
    "S2-E15": "S2E15",
    "S2-E18": "S2E18",
}


RUN_FILES = [

    "metrics.json",

    "embedding.npy",

    "modality_weights.npy",

    "pred_labels.npy",

    "gt_labels.npy",

    "coords.npy",

    "spot_ids.npy",
]


for dataset, folder in (
    DATASET_FOLDER.items()
):

    dst_dataset = (
        STAGE_DIR / folder
    )

    dst_dataset.mkdir(
        parents=True,
        exist_ok=True,
    )


    for seed in range(10):

        src_dir = (
            FORMAL_ROOT
            / f"{folder}_seed{seed}"
        )

        dst_dir = (
            dst_dataset
            / f"seed{seed}"
        )

        dst_dir.mkdir(
            parents=True,
            exist_ok=True,
        )


        for name in RUN_FILES:

            src = (
                src_dir / name
            )

            assert src.exists(), src

            shutil.copy2(
                src,
                dst_dir / name,
            )


README = """COSMOS formal baseline archive

Datasets
========
HLN-A1
HLN-D1
E18.5
S2-E15
S2-E18

Runs
====
10 training seeds per dataset
seeds = 0..9
50 total formal runs

Training
========
z_dim = 50
lr = 0.001
wnn_epoch = 500
total_epoch = 1000
max_patience_bef = 10
max_patience_aft = 30
min_stop = 200
spatial neighbors = 10

RNA+ADT
=======
RNA: normalize_per_cell + log1p
ADT: log1p
spatial_regularization_strength = 0.05
downstream clustering = Leiden

RNA+ATAC
========
spatial_regularization_strength = 0.01
downstream clustering = Louvain

Python 3.12 compatibility:
the legacy PyPI louvain backend was unavailable.
The Louvain algorithm was retained through
python-igraph community_multilevel.

A separate E18.5 audit verified:
- resolution changes cluster count
- fixed RNG is deterministic
- saved partition is exactly reproducible

Clustering
==========
50-NN graph

resolution scan:
0.05 to 2.50
step = 0.01

selection:
last resolution giving exact benchmark K

GT, ARI and NMI were not used to choose resolution.

Evaluation
==========
ARI:
sklearn adjusted_rand_score

NMI:
sklearn normalized_mutual_info_score
average_method = max

std:
ddof = 0

Final status
============
50/50 runs complete
all datasets have seeds 0-9
all 50 runs reached exact target K
"""


(
    STAGE_DIR
    / "README_ARCHIVE.txt"
).write_text(
    README,
    encoding="utf-8",
)


# ------------------------------------------------------------
# SHA256
# ------------------------------------------------------------

def sha256_file(path):

    h = hashlib.sha256()

    with path.open("rb") as f:

        while True:

            block = f.read(
                1024 * 1024
            )

            if not block:
                break

            h.update(block)

    return h.hexdigest()


manifest = []


for path in sorted(
    STAGE_DIR.rglob("*")
):

    if not path.is_file():
        continue

    if path.name == "SHA256SUMS.txt":
        continue

    rel = path.relative_to(
        STAGE_DIR
    )

    manifest.append(
        f"{sha256_file(path)}  {rel}"
    )


(
    STAGE_DIR
    / "SHA256SUMS.txt"
).write_text(
    "\n".join(manifest)
    + "\n",
    encoding="utf-8",
)


# ------------------------------------------------------------
# Structural audit
# ------------------------------------------------------------

assert len(
    list(
        STAGE_DIR.rglob(
            "metrics.json"
        )
    )
) == 50


assert len(
    list(
        STAGE_DIR.rglob(
            "embedding.npy"
        )
    )
) == 50


assert len(
    list(
        STAGE_DIR.rglob(
            "modality_weights.npy"
        )
    )
) == 50


assert len(
    list(
        STAGE_DIR.rglob(
            "pred_labels.npy"
        )
    )
) == 50


# ------------------------------------------------------------
# ZIP
# ------------------------------------------------------------

zip_path = shutil.make_archive(

    str(ZIP_BASE),

    "zip",

    root_dir=STAGE_DIR.parent,

    base_dir=STAGE_DIR.name,
)


zip_path = Path(
    zip_path
)


print("=" * 110)
print("COSMOS FINAL ARCHIVE")
print("=" * 110)

print(
    "Formal runs:",
    len(
        list(
            STAGE_DIR.rglob(
                "metrics.json"
            )
        )
    )
)

print(
    "SHA256 entries:",
    len(manifest)
)

print(
    "\nZIP:",
    zip_path
)

print(
    "ZIP size:",
    f"{zip_path.stat().st_size / 1024 / 1024:.2f} MB"
)

print(
    "\nPASS: COSMOS FINAL archive created."
)

COSMOS FINAL ARCHIVE
Formal runs: 50
SHA256 entries: 356

ZIP: /kaggle/working/COSMOS_5datasets_10seeds_FORMAL_FINAL.zip
ZIP size: 25.31 MB

PASS: COSMOS FINAL archive created.
